In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import pandas as pd
from glob import glob
import joblib
import json
import time
from copy import deepcopy
import os
import re
import numpy as np
import ee
import ast

In [4]:
agroclimaticZone_acronym_dict = {'Eastern Plateau & Hills Region': 'EPAHR',
                               'Southern Plateau and Hills Region': 'SPAHR',
                               'East Coast Plains & Hills Region': 'ECPHR',
                               'Western Plateau and Hills Region': 'WPAHR',
                               'Central Plateau & Hills Region': 'CPAHR',
                               'Lower Gangetic Plain Region': 'LGPR',
                                'Middle Gangetic Plain Region': 'MGPR',
                                'Eastern Himalayan Region': 'EHR',
                                'Western Himalayan Region': 'WHR',
                                'Upper Gangetic Plain Region': 'UGPR',
                                'Trans Gangetic Plain Region': 'TGPR',
                                'West Coast Plains & Ghat Region': 'WCPGR',
                                'Gujarat Plains & Hills Region': 'GPHR',
                                'Western Dry Region': 'WDR'}

In [5]:
# # This might need to be redefined
# best_month_dict = {'Eastern Plateau & Hills Region': 'cc_12',
#                    'Middle Gangetic Plain Region': 'cc_10',
#                    'Lower Gangetic Plain Region': 'cc_9',
#                    'Western Himalayan Region': 'cc_8',
#                    'Eastern Himalayan Region': 'cc_10',
#                    'Upper Gangetic Plain Region': 'cc_9',
#                    'Trans Gangetic Plain Region': 'cc_9',
#                    'Central Plateau & Hills Region': 'cc_7',
#                    'Western Plateau and Hills Region': 'cc_11',
#                    'Southern Plateau and Hills Region': 'cc_8',
#                    'East Coast Plains & Hills Region': 'cc_12'}

Rename the folders correctly before predicting. Sometimes some bug from GEE and Google Drive causes multiple copies of the same folder to be created and splits files between the 2 folders

## Rename

In [6]:
#agroclimatic_zone = 'Eastern Plateau & Hills Region' # DONE
# agroclimatic_zone = 'Southern Plateau and Hills Region' # DONE
# agroclimatic_zone = 'East Coast Plains & Hills Region' # DONE
# agroclimatic_zone = 'Western Plateau and Hills Region' # DONE
# agroclimatic_zone = 'Central Plateau & Hills Region' # DONE
# agroclimatic_zone = 'Lower Gangetic Plain Region' # DONE
# agroclimatic_zone = 'Middle Gangetic Plain Region' # DONE
# agroclimatic_zone = 'Eastern Himalayan Region' # DONE
# agroclimatic_zone = 'Western Himalayan Region' # DONE
# agroclimatic_zone = 'Upper Gangetic Plain Region' # DONE
#agroclimatic_zone = 'Trans Gangetic Plain Region' # DONE

In [7]:
df = pd.read_csv(f'drive/MyDrive/Anoushka/ACZ_Districts/{agroclimatic_zone}.csv')
dist_list = list(df['Name'])

In [8]:
# Set the list of years for which to rename folders
years = ['2022']

In [9]:
# Set the root directory to the specified folder
root_dir = '/content/drive/MyDrive/'
os.chdir(root_dir)

In [10]:
! pwd

/content/drive/MyDrive


In [11]:
# Get the current directory
current_directory = os.getcwd()
# List all folders in the current directory
folders = [f for f in os.listdir(current_directory) if os.path.isdir(os.path.join(current_directory, f))]
# Print the list of folders
# print("Folders in the current directory:")

# for folder in folders:
#     print(folder)

In [12]:
# Regular expression pattern to match folder names starting with a particular pattern
pattern = re.compile(r'^' + f'{agroclimaticZone_acronym_dict[agroclimatic_zone]}')

# Filter folders based on the pattern
matching_folders = [folder for folder in folders if pattern.match(folder)]

# Print the matching folders
print(f"Folders starting with '{agroclimaticZone_acronym_dict[agroclimatic_zone]}':")
for folder in matching_folders:
    print(folder)

Folders starting with 'WHR':
WHR_Doda_2018
WHR_Ganderbal_2018 (2)
WHR_Ganderbal_2018 (1)
WHR_Ganderbal_2018
WHR_Jammu_2018 (1)
WHR_Jammu_2018
WHR_Kargil_2018 (1)
WHR_Kargil_2018
WHR_Kathua_2018
WHR_Kishtwar_2018 (1)
WHR_Kishtwar_2018
WHR_Kulgam_2018 (2)
WHR_Kulgam_2018 (1)
WHR_Kulgam_2018
WHR_Kupwara_2018 (3)
WHR_Kupwara_2018 (2)
WHR_Kupwara_2018 (1)
WHR_Kupwara_2018
WHR_Leh (Ladakh)_2018
WHR_Poonch_2018 (2)
WHR_Poonch_2018 (1)
WHR_Poonch_2018
WHR_Pulwama_2018 (1)
WHR_Pulwama_2018
WHR_Rajouri_2018 (1)
WHR_Rajouri_2018
WHR_Ramban_2018 (1)
WHR_Ramban_2018
WHR_Reasi_2018 (1)
WHR_Reasi_2018
WHR_Samba_2018 (2)
WHR_Samba_2018 (1)
WHR_Samba_2018
WHR_Shupiyan_2018 (2)
WHR_Shupiyan_2018 (1)
WHR_Shupiyan_2018
WHR_Srinagar_2018 (1)
WHR_Srinagar_2018
WHR_Udhampur_2018 (3)
WHR_Udhampur_2018 (2)
WHR_Udhampur_2018 (1)
WHR_Udhampur_2018
WHR_Hoshiarpur_2018
WHR_Pathankot_2018
WHR_Rupnagar_2018
WHR_Sahibzada Ajit Singh Nagar_2018
WHR_Bijnor_2018 (1)
WHR_Bijnor_2018
WHR_Saharanpur_2018 (3)
WHR_Saharanpur

In [13]:
# Leh (Ladkh) is the only district encountered having special character '(' and ')' in it. That's why its handled in a special way as seen here

# Transfer files from duplicate folders to original folder

dist_num = 0
for district in dist_list:
    print(dist_num)
    for year in years:
        print(district, year)

        orig_district = district
        if district == 'Leh (Ladakh)':
            district = 'Leh \(Ladakh\)'

        pattern = re.compile(r'^' + agroclimaticZone_acronym_dict[agroclimatic_zone] + '_' + district + '_' + year)
        district_year_folders = [folder for folder in matching_folders if pattern.match(folder)]
        print(district_year_folders)
        district = orig_district

        while len(district_year_folders) > 1:
            print(len(district_year_folders))
            source_folder = district_year_folders[0]
            destination_folder = district_year_folders[1]

            files_to_move = os.listdir(source_folder)
            for file_name in files_to_move:
                source_path = os.path.join(source_folder, file_name)
                destination_path = os.path.join(destination_folder, file_name)
                os.rename(source_path, destination_path)

            del district_year_folders[0]

        current_folder_name = district_year_folders[0]
        new_folder_name = f'{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{district}_{year}'
        os.rename(current_folder_name, new_folder_name)

    dist_num += 1

0
Ambala 2022
['WHR_Ambala_2022']
1
Panchkula 2022
['WHR_Panchkula_2022 (1)', 'WHR_Panchkula_2022']
2
2
Yamunanagar 2022
['WHR_Yamunanagar_2022']
3
Bilaspur 2022
['WHR_Bilaspur_2022']
4
Chamba 2022
['WHR_Chamba_2022 (1)', 'WHR_Chamba_2022']
2
5
Hamirpur 2022
['WHR_Hamirpur_2022 (2)', 'WHR_Hamirpur_2022 (1)', 'WHR_Hamirpur_2022']
3
2
6
Kangra 2022
['WHR_Kangra_2022']
7
Kinnaur 2022
['WHR_Kinnaur_2022 (2)', 'WHR_Kinnaur_2022 (1)', 'WHR_Kinnaur_2022']
3
2
8
Kullu 2022
['WHR_Kullu_2022']
9
Lahul & Spiti 2022
['WHR_Lahul & Spiti_2022 (1)', 'WHR_Lahul & Spiti_2022']
2
10
Mandi 2022
['WHR_Mandi_2022']
11
Shimla 2022
['WHR_Shimla_2022']
12
Sirmaur 2022
['WHR_Sirmaur_2022']
13
Solan 2022
['WHR_Solan_2022 (1)', 'WHR_Solan_2022']
2
14
Una 2022
['WHR_Una_2022']
15
Anantnag 2022
['WHR_Anantnag_2022 (5)', 'WHR_Anantnag_2022 (4)', 'WHR_Anantnag_2022 (3)', 'WHR_Anantnag_2022 (2)', 'WHR_Anantnag_2022 (1)', 'WHR_Anantnag_2022']
6
5
4
3
2
16
Badgam 2022
['WHR_Badgam_2022 (2)', 'WHR_Badgam_2022 (1)', 'WHR

In [14]:
# Leh (Ladkh) is the only district encountered having special character '(' and ')' in it. That's why its handled in a special way as seen here

# Check all folders with more than 0 files
dist_num = 0
for district in dist_list:
    print(dist_num)
    for year in years:

        orig_district = district
        if district == 'Leh (Ladakh)':
            district = 'Leh \(Ladakh\)'

        pattern = re.compile(r'^' + agroclimaticZone_acronym_dict[agroclimatic_zone] + '_' + district + '_' + year)
        district_year_folders = [folder for folder in matching_folders if pattern.match(folder)]
        district = orig_district
        print(district_year_folders)

        for folder in district_year_folders:
            if len(os.listdir(folder)) > 0:
                print(folder)

    dist_num += 1

0
['WHR_Ambala_2022']
WHR_Ambala_2022
1
['WHR_Panchkula_2022 (1)', 'WHR_Panchkula_2022']
WHR_Panchkula_2022
2
['WHR_Yamunanagar_2022']
WHR_Yamunanagar_2022
3
['WHR_Bilaspur_2022']
WHR_Bilaspur_2022
4
['WHR_Chamba_2022 (1)', 'WHR_Chamba_2022']
WHR_Chamba_2022
5
['WHR_Hamirpur_2022 (2)', 'WHR_Hamirpur_2022 (1)', 'WHR_Hamirpur_2022']
WHR_Hamirpur_2022
6
['WHR_Kangra_2022']
WHR_Kangra_2022
7
['WHR_Kinnaur_2022 (2)', 'WHR_Kinnaur_2022 (1)', 'WHR_Kinnaur_2022']
WHR_Kinnaur_2022
8
['WHR_Kullu_2022']
WHR_Kullu_2022
9
['WHR_Lahul & Spiti_2022 (1)', 'WHR_Lahul & Spiti_2022']
WHR_Lahul & Spiti_2022
10
['WHR_Mandi_2022']
WHR_Mandi_2022
11
['WHR_Shimla_2022']
WHR_Shimla_2022
12
['WHR_Sirmaur_2022']
WHR_Sirmaur_2022
13
['WHR_Solan_2022 (1)', 'WHR_Solan_2022']
WHR_Solan_2022
14
['WHR_Una_2022']
WHR_Una_2022
15
['WHR_Anantnag_2022 (5)', 'WHR_Anantnag_2022 (4)', 'WHR_Anantnag_2022 (3)', 'WHR_Anantnag_2022 (2)', 'WHR_Anantnag_2022 (1)', 'WHR_Anantnag_2022']
WHR_Anantnag_2022
16
['WHR_Badgam_2022 (2)', '

In [15]:
# Change back to parent directory
os.chdir(os.path.dirname(os.getcwd()))

In [16]:
! pwd

/content/drive


## CCD

In [17]:
#agroclimatic_zone = 'Eastern Plateau & Hills Region'
# agroclimatic_zone = 'Southern Plateau and Hills Region'
# agroclimatic_zone = 'East Coast Plains & Hills Region'
# agroclimatic_zone = 'Western Plateau and Hills Region'
# agroclimatic_zone = 'Central Plateau & Hills Region'
# agroclimatic_zone = 'Lower Gangetic Plain Region'
# agroclimatic_zone = 'Middle Gangetic Plain Region'
# agroclimatic_zone = 'Eastern Himalayan Region'
agroclimatic_zone = 'Western Himalayan Region'
# agroclimatic_zone = 'Upper Gangetic Plain Region'
#agroclimatic_zone = 'Trans Gangetic Plain Region'

In [18]:
# df = pd.read_csv(f'MyDrive/Anoushka/ACZ_Districts/{agroclimatic_zone}.csv')
# dist_list = list(df['Name'])

In [19]:
# dist_list

In [20]:
agroclimatic_zone_model_path_mapping = {'Central Plateau & Hills Region': 'MyDrive/Anoushka/canopy_density/best_models/Central_Plateau_Hills_Region_toa_monthly_cover_72.joblib',
                                        'Lower Gangetic Plain Region': 'MyDrive/Anoushka/canopy_density/best_models/Lower_Gangetic_Plain_Region_toa_monthly_cover_67.joblib',
                                        'Middle Gangetic Plain Region': 'MyDrive/Anoushka/canopy_density/best_models/Middle_Gangetic_Plain_Region_toa_monthly_cover_75.joblib',
                                        'Eastern Himalayan Region': 'MyDrive/Anoushka/canopy_density/best_models/Eastern_Himalayan_Region_toa_monthly_cover_50.joblib',
                                        'Western Himalayan Region': 'MyDrive/Anoushka/canopy_density/best_models/Western_Himalayan_Region_toa_monthly_cover_50.joblib',
                                        'Upper Gangetic Plain Region': 'MyDrive/Anoushka/canopy_density/best_models/Upper_Gangetic_Plain_Region_toa_monthly_cover_40.joblib',
                                        'Trans Gangetic Plain Region': 'MyDrive/Anoushka/canopy_density/best_models/Trans_Gangetic_Plain_Region_toa_monthly_cover_83.joblib',
                                        'East Coast Plains & Hills Region': 'MyDrive/Anoushka/canopy_density/best_models/East_Coast_Plains_Hills_Region_toa_monthly_cover_40.joblib',
                                        'Eastern Plateau & Hills Region': 'MyDrive/Anoushka/canopy_density/best_models/Eastern_Plateau_Hills_Region_toa_monthly_cover_83.joblib',
                                        'Western Plateau and Hills Region': 'MyDrive/Anoushka/canopy_density/best_models/Western_Plateau_and_Hills_Region_toa_monthly_cover_80.joblib',
                                        'Southern Plateau and Hills Region': 'MyDrive/Anoushka/canopy_density/best_models/Southern_Plateau_and_Hills_Region_toa_monthly_cover_83.joblib'}

In [21]:
MODEL_PATH_cc = agroclimatic_zone_model_path_mapping[agroclimatic_zone]
model_cc = joblib.load(MODEL_PATH_cc)

In [22]:
if hasattr(model_cc, 'feature_names_in_'):
    features_cc = model_cc.feature_names_in_

In [23]:
seasons = ['kharif', 'rabi', 'zaid']

In [24]:
def add_s1_indices(df):
    for season in seasons:
        # SAR Indices
        df[f'VV_VH_Ratio_{season}'] = df[f'VV_{season}'] / df[f'VH_{season}']
        df[f'VH_VV_Ratio_{season}'] = df[f'VH_{season}'] / df[f'VV_{season}']
        df[f'SAR_NDVI_{season}'] = (df[f'VH_{season}'] - df[f'VV_{season}']) / (df[f'VH_{season}'] + df[f'VV_{season}'])
        df[f'SAR_DVI_{season}'] = df[f'VH_{season}'] - df[f'VV_{season}']
        df[f'SAR_SVI_{season}'] = df[f'VH_{season}'] + df[f'VV_{season}']
        df[f'SAR_RDVI_{season}'] = (df[f'VH_{season}'] / df[f'VV_{season}']) - (df[f'VV_{season}'] / df[f'VH_{season}'])
        df[f'SAR_NRDVI_{season}'] = ((df[f'VH_{season}']/df[f'VV_{season}'] - df[f'VV_{season}']/df[f'VH_{season}']) / (df[f'VH_{season}']/df[f'VV_{season}'] + df[f'VV_{season}']/df[f'VH_{season}']))
        df[f'SAR_SSDVI_{season}'] = df[f'VH_{season}']**2 - df[f'VV_{season}']**2

def add_s2_indices(df):
    for season in seasons:
        # Optical Indices
        df[f'NDVI_{season}'] = (df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + df[f'B4_{season}'])
        df[f'NDWI_{season}'] = (df[f'B8_{season}'] - df[f'B12_{season}']) / (df[f'B8_{season}'] + df[f'B12_{season}'])
        df[f'EVI_{season}'] = (2.5 * ((df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + 6*df[f'B4_{season}'] - 7.5*df[f'B2_{season}'] + 1)))
        df[f'OSAVI_{season}'] = (df[f'B8_{season}'] - df[f'B4_{season}']) / (df[f'B8_{season}'] + df[f'B4_{season}'] + 0.16)
        df[f'ARVI_{season}'] = (df[f'B8_{season}'] - 2*df[f'B4_{season}'] + df[f'B2_{season}']) / (df[f'B8_{season}'] + 2*df[f'B4_{season}'] + df[f'B2_{season}'])
        df[f'VARI_{season}'] = (df[f'B3_{season}'] - df[f'B4_{season}']) / (df[f'B3_{season}'] + df[f'B4_{season}'] - df[f'B2_{season}'])

In [25]:
def get_csv_data(fileName):
    data = pd.DataFrame()
    try:
        data = pd.read_csv(fileName)
    except Exception as exp:
        print("Error reading file ", fileName, " - ", exp)
    return data

In [26]:
# For Canopy Cover
def pipeline(fileName):

    print(fileName)

    df = get_csv_data(fileName)

    if (len(df) == 0):
        return df

    add_s1_indices(df)
    add_s2_indices(df)

    geoList = list(df['.geo'])
    res_df = pd.DataFrame()
    res_df['.geo'] = geoList

    for month in range(1,13):
        df['month_sin'] = [np.sin(2 * np.pi * month / 12)] * len(df)
        df['month_cos'] = [np.cos(2 * np.pi * month / 12)] * len(df)

        test_df = df[features_cc]
        pred_y_cc = list(model_cc.predict(test_df))
        res_df[f'cc_{month}'] = pred_y_cc

    return res_df

In [27]:
for year in years:
    dist_num = 0
    for district in dist_list:
        # if dist_num < 100:
        #     dist_num += 1
        #     continue
        print('\n', dist_num, district, year)
        dist_data_path = f'MyDrive/{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{district}_{year}/'
        files = glob(dist_data_path + "*.csv")
        print("no. of files:", len(files), 'at', dist_data_path, '\n')
        merged_df = pd.DataFrame()
        for fileName in files:
            df = pipeline(fileName)

            merged_df = pd.concat([merged_df, df])

        merged_df.to_csv(f'MyDrive/Anoushka/canopy_density/results/{agroclimatic_zone}_{district}_{year}_monthly_cc.csv', index=False)
        dist_num += 1


 0 Ambala 2022
no. of files: 3 at MyDrive/WHR_Ambala_2022/ 

MyDrive/WHR_Ambala_2022/Ambala_0_2022.csv
MyDrive/WHR_Ambala_2022/Ambala_1_2022.csv
MyDrive/WHR_Ambala_2022/Ambala_2_2022.csv

 1 Panchkula 2022
no. of files: 9 at MyDrive/WHR_Panchkula_2022/ 

MyDrive/WHR_Panchkula_2022/Panchkula_0_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_1_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_6_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_7_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_2_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_3_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_5_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_9_2022.csv
MyDrive/WHR_Panchkula_2022/Panchkula_8_2022.csv

 2 Yamunanagar 2022
no. of files: 7 at MyDrive/WHR_Yamunanagar_2022/ 

MyDrive/WHR_Yamunanagar_2022/Yamunanagar_1_2022.csv
MyDrive/WHR_Yamunanagar_2022/Yamunanagar_3_2022.csv
MyDrive/WHR_Yamunanagar_2022/Yamunanagar_4_2022.csv
MyDrive/WHR_Yamunanagar_2022/Yamunanagar_0_2022.csv
MyDrive/WHR_Yamunanagar_2022/Yam

In [28]:
# dist_data_path = f'drive/MyDrive/{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{district}_{year}/'
# files = glob(dist_data_path + "*.csv")
